In [1]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
import torch
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
import pandas as pd
import numpy as np
import os
import pickle
import glob

In [2]:
artnet_2025 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_sold.xlsx")

In [3]:
artnet_2025.columns

Index(['lot id', 'artwork id', 'artist id', 'sale date', 'artist modifier',
       'first', 'last', 'nationality', 'year born', 'year died', 'title',
       'workyear modifier', 'workyear from', 'workyear to', 'est lo', 'est hi',
       'sale price', 'currency', 'currency exchange rate', 'est lo usd',
       'est hi usd', 'sale price usd', 'priceStatus', 'pricePhrase',
       'has_image'],
      dtype='object')

In [4]:
sample_image = Image.open(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\2706622.jpg")

In [5]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
for i in range(artnet_2025.shape[0]):
    filename = f'{artnet_2025.iloc[i]["artwork id"]}.jpg'
    try:
        image = Image.open(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{filename}")
        if image==sample_image:
            artnet_2025.loc[i, "has_image"]=0
    except:
        artnet_2025.loc[i, "has_image"]=0
    if i % 10000 == 0:
        print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-10-28 10:01:34: Start
2025-10-28 10:01:34: 0
2025-10-28 10:02:00: 10000
2025-10-28 10:02:23: 20000
2025-10-28 10:02:44: 30000
2025-10-28 10:03:06: 40000
2025-10-28 10:03:27: 50000
2025-10-28 10:03:48: 60000
2025-10-28 10:04:17: 70000
2025-10-28 10:04:40: 80000
2025-10-28 10:05:01: 90000
2025-10-28 10:05:24: 100000
2025-10-28 10:05:45: 110000
2025-10-28 10:06:12: 120000
2025-10-28 10:06:42: 130000
2025-10-28 10:07:10: 140000
2025-10-28 10:07:41: 150000
2025-10-28 10:08:11: 160000
2025-10-28 10:08:41: 170000
2025-10-28 10:09:10: 180000
2025-10-28 10:09:39: 190000
2025-10-28 10:10:07: 200000
2025-10-28 10:10:35: 210000
2025-10-28 10:11:22: 220000
2025-10-28 10:12:21: 230000
2025-10-28 10:13:17: 240000
2025-10-28 10:14:12: 250000
2025-10-28 10:15:05: 260000
2025-10-28 10:16:01: 270000
2025-10-28 10:16:56: 280000
2025-10-28 10:17:51: 290000
2025-10-28 10:18:47: 300000
2025-10-28 10:19:43: 310000
2025-10-28 10:20:39: 320000
2025-10-28 10:21:35: 330000
2025-10-28 10:22:30: 340000
2025-10

In [8]:
np.sum(artnet_2025["has_image"])

np.int64(700367)

In [11]:
artnet_2025_valid = artnet_2025[artnet_2025["has_image"]==1]

In [12]:
artnet_2025_valid.to_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_image_cleaned.xlsx",index=False)

## Embedding

In [2]:
artnet_2025= pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe.xlsx")

In [4]:
artnet_2025.shape

(700367, 25)

In [3]:
#number_size = int(sys.argv[1])
#range_start = int(sys.argv[2])
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [4]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",use_fast=True)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [6]:
def convert_one_row(model,i, image_file,device):
    try:
        image = Image.open(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{image_file}")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        image_features = image_features.cpu().numpy()
    except Exception as e:
        print(f"Error processing {image_file}: {e}")
        image_features = np.zeros([1,512])
        
    return i, image_features

In [ ]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, f'{artnet_2025.iloc[i]["artwork id"]}.jpg',device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, image_features = fut.result()
        embeddings[i-range_start] =image_features
        if i % 10000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
np.save(f"Result/clip_embeddings_{range_start}.npy", embeddings)

In [9]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2025.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    embeddings = np.zeros(N, dtype=object)
    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {
            ex.submit(convert_one_row, model,i, f'{artnet_2025.iloc[i]["artwork id"]}.jpg',device): i
            for i in range(range_start,range_end)
        }
        for fut in as_completed(futures):
            i, image_features = fut.result()
            embeddings[i-range_start] =image_features
            if i % 10000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
    np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding\\clip_embeddings_{range_start}.npy", embeddings)
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2025.shape[0])
    N = min(number_size, artnet_2025.shape[0]-range_start)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-10-30 15:53:10: Start
2025-10-30 15:53:10: Now at 0
2025-10-30 15:53:36: 0
2025-10-30 15:54:45: 10000
2025-10-30 15:56:07: 20000
2025-10-30 15:57:29: 30000
2025-10-30 15:58:40: 40000
2025-10-30 16:00:01: 50000
2025-10-30 16:01:23: 60000
2025-10-30 16:02:45: 70000
2025-10-30 16:04:07: 80000
2025-10-30 16:05:28: 90000
2025-10-30 16:06:52: Saving embeddings
2025-10-30 16:06:52: Now at 100000
2025-10-30 16:07:37: 100000
2025-10-30 16:08:27: 110000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-10-30 16:09:48: 120000
2025-10-30 16:11:10: 130000
2025-10-30 16:12:33: 140000
2025-10-30 16:13:56: 150000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-10-30 16:15:19: 160000
2025-10-30 16:16:41: 170000


...

2025-10-30 16:39:13: 330000
2025-10-30 16:40:38: 340000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-10-30 16:42:03: 350000
2025-10-30 16:43:28: 360000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-10-30 16:44:52: 370000
2025-10-30 16:45:07: Saving embeddings
2025-10-30 16:45:07: Ends


In [10]:
artnet_2025.shape

(371778, 26)

## Merge all Outputs

In [11]:
folder_path = "D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding"
npy_files = glob.glob(f"{folder_path}/*.npy")

In [12]:
arrays = [np.load(f, allow_pickle=True) for f in npy_files]
big_array = np.concatenate(arrays, axis=0)
np.save("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\clip_embedding.npy", big_array)

# Formal 2024

In [4]:
artnet_2024 = pd.read_excel("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Artnet_Europe_2024_merged.xlsx")

In [5]:
artnet_2024.shape

(355551, 16)

In [ ]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2025.shape[0])
N = min(number_size, artnet_2025.shape[0]-range_start)

In [6]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32",use_fast=True)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [8]:
def convert_one_row(model,i, image_file,device):
    try:
        image = Image.open(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\ArtNet\\Images\\{image_file}")
        inputs = processor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            image_features = model.get_image_features(**inputs)
        image_features = image_features / image_features.norm(p=2, dim=-1, keepdim=True)
        image_features = image_features.cpu().numpy()
    except Exception as e:
        print(f"Error processing {image_file}: {e}")
        image_features = np.zeros([1,512])
        
    return i, image_features

In [ ]:
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
embeddings = np.zeros(N, dtype=object)
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {
        ex.submit(convert_one_row, model,i, f'{artnet_2025.iloc[i]["artwork id"]}.jpg',device): i
        for i in range(range_start,range_end)
    }
    for fut in as_completed(futures):
        i, image_features = fut.result()
        embeddings[i-range_start] =image_features
        if i % 10000 == 0:
            print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

In [ ]:
np.save(f"Result/clip_embeddings_{range_start}.npy", embeddings)

In [10]:
number_size = 100000
range_start = 0
range_end = min(range_start+number_size,artnet_2024.shape[0])
N = min(number_size, artnet_2024.shape[0]-range_start)
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Start")
while range_start < artnet_2024.shape[0]:
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Now at {range_start}")
    embeddings = np.zeros(N, dtype=object)
    with ThreadPoolExecutor(max_workers=32) as ex:
        futures = {
            ex.submit(convert_one_row, model,i, f'{artnet_2024.iloc[i]["artwork id"]}.jpg',device): i
            for i in range(range_start,range_end)
        }
        for fut in as_completed(futures):
            i, image_features = fut.result()
            embeddings[i-range_start] =image_features
            if i % 10000 == 0:
                print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: {i}")
    np.save(f"D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding_2024\\clip_embeddings_{range_start}.npy", embeddings)
    range_start = range_start + N
    range_end = min(range_start+number_size,artnet_2024.shape[0])
    N = min(number_size, artnet_2024.shape[0]-range_start)
    print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Saving embeddings")
    
print(f"{datetime.now():%Y-%m-%d %H:%M:%S}: Ends")

2025-12-08 03:21:23: Start
2025-12-08 03:21:23: Now at 0
2025-12-08 03:21:34: 0
2025-12-08 03:22:55: 10000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-12-08 03:24:17: 20000
2025-12-08 03:25:38: 30000
2025-12-08 03:26:59: 40000
2025-12-08 03:28:19: 50000
2025-12-08 03:29:41: 60000
2025-12-08 03:31:02: 70000
2025-12-08 03:32:22: 80000
2025-12-08 03:33:43: 90000
2025-12-08 03:35:05: Saving embeddings
2025-12-08 03:35:05: Now at 100000
2025-12-08 03:35:52: 100000
2025-12-08 03:36:39: 110000
2025-12-08 03:38:01: 120000
2025-12-08 03:39:24: 130000
2025-12-08 03:40:47: 140000
2025-12-08 03:42:10: 150000
2025-12-08 03:43:33: 160000
2025-12-08 03:44:55: 170000
2025-12-08 03:46:18: 180000
2025-12-08 03:47:40: 190000
2025-12-08 03:49:03: Saving embeddings
2025-12-08 03:49:03: Now at 200000
2025-12-08 03:49:40: 200000
2025-12-08 03:50:38: 210000
2025-12-08 03:52:01: 220000
2025-12-08 03:53:23: 230000
2025-12-08 03:54:46: 240000
2025-12-08 03:56:08: 250000
2025-12-08 03:57:31: 260000
2025-12-08 03:58:53: 270000
2025-12-08 04:00:16: 280000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


2025-12-08 04:01:38: 290000
2025-12-08 04:03:02: Saving embeddings
2025-12-08 04:03:02: Now at 300000
2025-12-08 04:03:41: 300000
2025-12-08 04:04:31: 310000
2025-12-08 04:05:53: 320000
2025-12-08 04:07:16: 330000
2025-12-08 04:08:39: 340000
2025-12-08 04:10:01: 350000


The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is ambiguous. Got image shape torch.Size([3, 1, 1]). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.
The channel dimension is amb

2025-12-08 04:10:47: Saving embeddings
2025-12-08 04:10:47: Ends


# Merging

In [11]:
folder_path = "D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\Embedding_2024"
npy_files = glob.glob(f"{folder_path}/*.npy")

In [12]:
arrays = [np.load(f, allow_pickle=True) for f in npy_files]
big_array = np.concatenate(arrays, axis=0)
np.save("D:\\MissTiny\\GitHub\\Creativity_Artnet\\Datasets\\clip_embedding_2024.npy", big_array)